# 원신 캐릭터 정보 RAG — 보조 지표: 정답률 · 거절 정확도 · 환각률

이미 저장된 DEV 결과 JSON(`results/dev_*.json`)을 읽어서 보조 지표 3개를 계산하고 JSON으로 저장합니다.
모델을 다시 돌리지 않으므로 **GPU가 필요 없습니다** (CPU 런타임).

| 지표 | 뜻 | 좋은 방향 | 참고 |
|---|---|---|---|
| 정답률 | 답이 정답과 완전히 같은 문항 비율 (F1 = 1) | 높을수록 | 4주차 Ex17 (맞힌 개수 세기) |
| 거절 정확도 | 데이터에 없는 질문(unknown)에 빈 답 `[]`을 낸 비율 | 높을수록 | 6주차 Ex12·13 (근거 없는 질문 처리) |
| 환각률 | 답을 낸 문항 중, **근거에 없는 이름·값**이 섞인 문항 비율 | 낮을수록 | 5주차 노트 16쪽 근거 충실성(Faithfulness), 5주차 Ex01 (집합) |

- "근거에 있다" = 모델이 받은 근거 문장(`이름: 원소 X / 무기 Y / 5성 / 지역 Z`)의 이름·값과 표기만 맞추면 같은 것.
  일본어가 섞인 이름(`라이デン 쇼군`)이나 오타도 근거에 없는 것으로 센다.
- 근거는 결과 JSON에 없어서 같은 코드(`core/` 라우터·검색)로 다시 찾는다. DEV는 학습용 캐릭터에서만 찾는다.
- 후처리(`_post`)는 근거에 있는 이름만 남기므로 환각률이 0이 된다. 환각률은 후처리 전 결과를 본다.

**사용법**
1. (먼저) 노트북 01 셀 10-1에서 비교할 방법의 DEV 결과를 만들어 둔다. 예: `DEV_METHODS = ["baseline", "prompt_v2", "rag"]`
2. 이 노트북은 CPU 런타임으로 셀 1~4를 차례로 실행
3. 결과: 드라이브 `results/aux/aux_{실험 ID}.json`, 셀 4가 표로 비교한다

In [ ]:
# 1. 설정: 저장소 코드(core/) 받기 + 드라이브 연결 + 캐릭터 데이터 불러오기 (노트북 01, 02와 같다)
import hashlib
import json
import os
import re
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

REPO = "genshin-build-alchemist/genshin-build-alchemist"
REPO_BRANCH = "main"       # PR 브랜치를 시험할 때만 바꾼다
IN_COLAB = "google.colab" in sys.modules


def github_token():
    """코랩 보안 비밀(왼쪽 열쇠 아이콘)의 GITHUB_TOKEN. 저장소가 공개면 없어도 된다."""
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return os.environ.get("GITHUB_TOKEN")


def run_git(args, token):
    """git 실행. 출력에 토큰이 찍히지 않게 가린다."""
    r = subprocess.run(["git"] + args, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(token, "***") if token else r.stdout + r.stderr
    if r.returncode != 0:
        raise RuntimeError("git 실패:\n" + out + "\n코랩 보안 비밀에 GITHUB_TOKEN을 넣었는지 확인하세요 (notebooks/README.md).")


if IN_COLAB:
    REPO_DIR = Path("/content/genshin-build-alchemist")
    token = github_token()
    url = f"https://{'x-access-token:' + token + '@' if token else ''}github.com/{REPO}.git"
    if (REPO_DIR / ".git").exists():
        run_git(["-C", str(REPO_DIR), "fetch", "--depth", "1", url, REPO_BRANCH], token)
        run_git(["-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"], token)
    else:
        run_git(["clone", "--depth", "1", "--branch", REPO_BRANCH, url, str(REPO_DIR)], token)
        run_git(["-C", str(REPO_DIR), "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], token)
else:   # 로컬: 이 노트북이 들어 있는 저장소를 쓴다 (GBA_REPO 환경변수로 지정 가능)
    REPO_DIR = Path(os.environ.get("GBA_REPO") or next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "core").is_dir()))
sys.path.insert(0, str(REPO_DIR))
commit = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip() or "unknown"
CODE_VERSION = f"{REPO_BRANCH}@{commit}"    # 결과 JSON에 남긴다 (어느 코드로 돌렸는지)
print(f"저장소 코드: {REPO_DIR} ({CODE_VERSION})")

from core.kb import load_kb                                   # noqa: E402
from core.retrieval import retrieve as core_retrieve           # noqa: E402
from core.router import Router                                # noqa: E402

# 팀 드라이브 폴더 (공유받은 폴더를 '내 드라이브'에 바로가기로 추가해 두어야 보인다)
TEAM_DIR = Path("/content/drive/MyDrive/Genshin Build Alchemist (팀)")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
if TEAM_DIR.exists():
    RESULT_DIR = TEAM_DIR / "results"
    print(f"팀 드라이브 사용: {TEAM_DIR}")
else:
    RESULT_DIR = Path(os.environ.get("GBA_RESULTS", "results"))
    print(f"팀 드라이브 폴더가 없어서 {RESULT_DIR} 폴더를 씁니다.")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

KB = load_kb(REPO_DIR / "data")
characters = KB.characters
print(f"캐릭터 {len(characters)}명 불러옴")

In [ ]:
# 2. 질문 + 근거 낱말 찾기
from core.answer import canonical, evidence_words   # noqa: E402  (후처리와 같은 기준으로 표기를 맞춘다)

with open(REPO_DIR / "data" / "eval" / "questions_info_v2.json", encoding="utf-8") as f:
    frozen = json.load(f)
Q_BY_ID = {q["id"]: q for q in frozen["questions"]}
EVAL_CHARACTERS = set(frozen["eval_characters"])
TRAIN_POOL = [c for c in characters if c["name"] not in EVAL_CHARACTERS]
ROUTER = Router(KB)


def evidence_set(q):
    """질문 -> 근거에 있는 낱말 집합 (이름, 원소, 무기, 성급, 지역). 학습용 질문은 학습용 캐릭터에서만 찾는다."""
    pool = TRAIN_POOL if q["split"] == "train" else characters
    found = core_retrieve(ROUTER.route(q["question"]), KB, pool)
    texts = [e["text"] for e in found if e["kind"] == "character"]
    return {canonical(w) for w in evidence_words(texts)}


def clean(p):
    """답 앞뒤에 남은 [ ] " ' 를 지운다 (모델 답이 중간에 잘려 목록이 깨진 경우)."""
    return str(p).strip().strip('[]"\' ')


# 확인: 근거 낱말 예시
q = next(q for q in Q_BY_ID.values() if q["split"] == "train" and q["type"] == "lookup")
print(q["question"], "->", sorted(evidence_set(q)))

In [ ]:
# 3. 보조 지표 3개 계산 -> results/aux/aux_{실험 ID}.json 저장
AUX_DIR = RESULT_DIR / "aux"
AUX_DIR.mkdir(parents=True, exist_ok=True)


def aux_metrics(experiment_id):
    path = RESULT_DIR / f"{experiment_id}.json"
    if not path.exists():
        print(f"건너뜀: {path.name} 가 없습니다")
        return None
    with open(path, encoding="utf-8") as f:
        src = json.load(f)

    correct = 0          # 정답률: 완전히 맞힌 문항 수
    unknown_total = 0    # 거절 정확도: 모르는 질문 수
    refused = 0          #              그중 빈 답 [] 을 낸 수
    answered = 0         # 환각률: 답을 낸 문항 수
    hallucinated = 0     #         그중 근거에 없는 답이 섞인 문항 수
    hallucinations = []  # 근거에 없는 답 목록 (오답 분석용)

    for x in src["per_question"]:
        pred = [clean(p) for p in x["pred"]]
        pred = [p for p in pred if p]

        if x["f1"] == 1.0:
            correct = correct + 1

        if x["type"] == "unknown":
            unknown_total = unknown_total + 1
            if not pred:
                refused = refused + 1

        if pred:
            answered = answered + 1
            allowed = evidence_set(Q_BY_ID[x["id"]])
            not_in_evidence = [p for p in pred if canonical(p) not in allowed]
            if not_in_evidence:
                hallucinated = hallucinated + 1
                hallucinations.append({"id": x["id"], "question": x["question"],
                                       "pred": pred, "not_in_evidence": not_in_evidence})

    n = len(src["per_question"])
    result = {
        "experiment_id": f"aux_{experiment_id}",
        "source_experiment": experiment_id,
        "method": src["method"],
        "model": src["model"],
        "dataset_version": src["dataset_version"],
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "config": {"code_version": CODE_VERSION, "notebook": "03_aux_metrics"},
        "metrics": {
            "n": n,
            "accuracy": round(correct / n, 4),
            "refusal_accuracy": round(refused / unknown_total, 4) if unknown_total else None,
            "hallucination_rate": round(hallucinated / answered, 4) if answered else None,
            "answered": answered,
            "hallucinated_questions": hallucinated,
        },
        "hallucinations": hallucinations,
        "notes": "보조 지표: 정답률(F1=1 비율), 거절 정확도(unknown에 [] 비율), 환각률(답한 문항 중 근거에 없는 답이 섞인 비율)",
    }
    with open(AUX_DIR / f"aux_{experiment_id}.json", "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    return result

In [ ]:
# 4. 실행 + 비교표 (DEV, T4 3B fp16 기준. 단계 순서대로)
import pandas as pd   # noqa: E402

TARGETS = ["dev_baseline", "dev_prompt_v2", "dev_rag", "dev_rag_v3", "dev_rag_v4", "dev_rag_v4_post"]

table = []
for name in TARGETS:
    r = aux_metrics(name)
    if r is None:
        continue
    m = r["metrics"]
    table.append({"결과": name, "정답률": m["accuracy"], "거절 정확도": m["refusal_accuracy"],
                  "환각률": m["hallucination_rate"], "환각 문항 수": f"{m['hallucinated_questions']} / {m['answered']}"})
    for h in r["hallucinations"]:
        print(f"[{name}] {h['question']} -> 근거에 없는 답 {h['not_in_evidence']}")

pd.DataFrame(table)